<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/Copy_of_Final_MC06_Playground_to_SDK_2026_08_22_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MC06 — Building Your First Chatbot
**Decoding Data Science | AI Residency**

Here's what we're doing today, in order:

1. Build and test a simple chatbot in the **OpenAI Playground** — no code, just conversation.
2. Save it there, which gives us a **Prompt ID** — and run that exact prompt right here in this notebook.
3. Then rewrite the same chatbot **directly in code** using the OpenAI SDK — this is what you'll do once you're building a real application.

Everything below happens in **this one notebook**, top to bottom. Just run each cell in order.

## Step 0 — Install the OpenAI library

In [1]:
!pip install openai --quiet

## Step 1 — Store your API key (the safe way)

**Never type your API key directly into a notebook cell.** If you do, and you ever share this notebook or push it to GitHub, your key is exposed to the world. Colab has a built-in, private vault for exactly this — called **Secrets**. Here's how to set it up, slowly:

1. Look at the **left sidebar** of this Colab window. You'll see a few icons stacked vertically — folder, magnifying glass, puzzle piece, and a **key icon (🔑)**. Click the **key icon**.
2. A panel opens. Click **"+ Add new secret"**.
3. In the **Name** field, type exactly: `OPENAI_API_KEY` (capital letters, underscores — must match exactly).
4. In the **Value** field, paste your actual OpenAI API key (the one starting with `sk-...`).
5. There's a toggle next to your secret labeled **"Notebook access"** — make sure it's **switched ON**. If it's off, this notebook can't read it, even though it's saved.
6. That's it — close the panel and run the cell below.

Your key now lives in Colab's private vault, tied to your Google account — not typed anywhere in this notebook, not visible if you share the notebook with someone else.

In [2]:
from google.colab import userdata
from openai import OpenAI

# This line reads your key from the Secrets vault — it never appears as text here
client = OpenAI(api_key=userdata.get('openai'))

print('Client ready — your key was loaded securely.')

Client ready — your key was loaded securely.


**If you get an error here:** it almost always means one of two things — either the secret name isn't exactly `OPENAI_API_KEY` (check spelling and capitals), or the "Notebook access" toggle next to it is switched off. Go back to Step 1 and check both.

## Part A — Run the chatbot you built in Playground

**What we just did together in the Playground:**
1. Opened the OpenAI Playground.
2. Wrote simple instructions for our chatbot — what it should do, how it should behave.
3. Tested it with a message, read the response, tightened the instructions, tested again.
4. Once it worked well, clicked **Save** — which gave us a **Prompt ID** (something like `pmpt_XXXXXXXXXXXXXXXXXXXX`).

That Prompt ID is now a saved, reusable object. Instead of retyping our instructions in code, we just point to it. **Paste your own Prompt ID below** (copy it from Playground) before running this cell.

In [4]:
# Paste the Prompt ID you copied from Playground here, between the quotes
#PROMPT_ID = "pmpt_REPLACE_WITH_YOUR_ID"

PROMPT_ID = "pmpt_6a8972ced1b081908b2e5fbd04338f68042704ee7d91e718"

response = client.responses.create(
    prompt={
        "id": PROMPT_ID,
        "version": "1"
    },
    input="Hi! Can you help me understand what you can do?"
)

print(response.output_text)

Reasoning: I can help you learn Python by explaining ideas clearly, breaking programming problems into small steps, and helping you write or debug code. You can ask beginner questions (variables, loops, functions), intermediate topics (classes, files, APIs), or more advanced subjects (decorators, async programming, algorithms, testing).

Examples of things I can help with:

- Explain Python concepts: “What is a dictionary?” or “How do `*args` and `**kwargs` work?”
- Write code step by step: “Make a to-do list program.”
- Debug errors: Paste your code and the full error message.
- Improve code: “How can I make this function cleaner or faster?”
- Practice problems: “Give me an easy loop exercise.”
- Work with data: CSV files, JSON, pandas, lists, dictionaries, etc.
- Learn best practices: functions, naming, testing, virtual environments, and project structure.

```python
# For example, I can explain this simple function:
def greet(name):
    return f"Hello, {name}!"

print(greet("Ada"))


If this printed a response in the style of what you saw in Playground — that's it, working. You've just run your Playground-tested prompt from Python code, with zero extra setup.

**One more thing worth knowing:** if you go back into Playground later and improve this prompt, you don't need to touch this notebook at all — the same Prompt ID will automatically use your improved version.

## Part B — Now, the same chatbot written directly in code

This time, instead of pointing to a saved prompt, we write the instructions ourselves, right here in Python. This is the pattern you'll use once you're building a real application — the instructions live in your code, not in a separate saved prompt.

Everything else about the call looks almost identical to Part A.

In [ ]:
# Our instructions, written directly in code this time
SYSTEM_INSTRUCTIONS = (
    "You are a helpful assistant for Decoding Data Science students. "
    "Answer clearly and concisely. If you don't know something, say so."
)

response = client.responses.create(
    model="gpt-5.4",
    reasoning={"effort": "medium"},
    instructions=SYSTEM_INSTRUCTIONS,
    input="Hi! Can you help me understand what you can do?"
)

print(response.output_text)

Absolutely — I can help with a wide range of things, especially around learning and problem-solving.

Here’s what I can do well:

### For data science and coding
- Explain concepts in simple terms
- Help with Python, SQL, statistics, and machine learning
- Debug code and explain errors
- Walk through assignments or practice problems step by step
- Help interpret results, graphs, and model outputs
- Suggest study plans or learning resources

### For writing and communication
- Summarize articles, notes, or research
- Rewrite text to be clearer or more professional
- Help draft emails, reports, or presentations
- Explain technical ideas for different audiences

### For studying and learning
- Answer questions interactively
- Quiz you on topics
- Create examples and practice exercises
- Break down difficult material into smaller pieces

### For general productivity
- Brainstorm ideas
- Compare options
- Help organize projects or to-do lists
- Assist with planning and decision-making

### 

**What's `reasoning={"effort": "medium"}`?** We're using a reasoning model, and this one line controls how much the model "thinks" before answering — `"low"` for quick answers, `"medium"` for balanced, `"high"` when the task needs deeper thinking. That's the only setting we need to know for today — keep it on `"medium"` unless we say otherwise.

## Part C — Make it a real, back-and-forth chatbot

Right now, each call answers one message and forgets it ever happened. A real chatbot needs to remember the conversation. We do that by keeping a running list of every message — yours and the bot's — and sending the whole list back each time.

Run the cell below, then type in the box that appears. Type `quit` to stop.

In [ ]:
def chat_with_bot():
    print("DDS Chatbot — type 'quit' to exit\n")
    conversation = []

    while True:
        user_input = input("You: ")
        if user_input.strip().lower() == "quit":
            print("Bot: Goodbye!")
            break

        conversation.append({"role": "user", "content": user_input})

        response = client.responses.create(
            model="gpt-5.4",
            reasoning={"effort": "medium"},
            instructions=SYSTEM_INSTRUCTIONS,
            input=conversation
        )

        reply = response.output_text
        print("Bot:", reply, "\n")
        conversation.append({"role": "assistant", "content": reply})

chat_with_bot()

Try asking it something, then follow up with a question that only makes sense if it remembers your first message (e.g. "what did I just ask you?"). If it answers correctly, the memory is working — that's because we're sending the *entire* conversation list back each time, not just your latest message.

## Recap
- **Part A:** ran a prompt you built and saved in Playground, using its Prompt ID.
- **Part B:** rewrote the same thing with instructions directly in code.
- **Part C:** turned it into a real, memory-holding chatbot.

This chat loop is the foundation — everything we build next (function calling, data integration) extends this exact skeleton.

##Adding UI Interface using Gradio

In [ ]:
import gradio as gr
print(gr.__version__)

6.24.0


In [ ]:
SYSTEM_INSTRUCTIONS

"You are a helpful assistant for Decoding Data Science students. Answer clearly and concisely. If you don't know something, say so."

In [ ]:
import gradio as gr

def chatbot_response(message):
    response = client.responses.create(
        model="gpt-5.4",
        reasoning={"effort": "medium"},
        instructions=SYSTEM_INSTRUCTIONS,
        input=message
    )
    return response.output_text

demo = gr.Interface(
    fn=chatbot_response,
    inputs=gr.Textbox(label="Your message"),
    outputs=gr.Textbox(label="DDS Chatbot"),
    title="DDS Chatbot"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e6cde08d49f2e6a7cf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
##with chatinterface
import gradio as gr

def chatbot_response(message, history):
    conversation = history + [{"role": "user", "content": message}]

    response = client.responses.create(
        model="gpt-5.4",
        reasoning={"effort": "medium"},
        instructions=SYSTEM_INSTRUCTIONS,
        input=conversation
    )
    return response.output_text

demo = gr.ChatInterface(fn=chatbot_response, title="DDS Chatbot")
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dbfc91982a9863a4d8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
